# 56 — Adversarial resume pair generation (controlled counterfactuals)

**Role:** Build the **pair registry** for the adversarial evaluation track by applying **minimal, rule-based** edits to **real test-split resumes** only.

**Consumption:** Reads `adversarial_benchmark_spec.json` from notebook **55** (design). **Does not** train models or run final benchmark metrics.

**Outputs:**
- `notebooks/results/adversarial_resume_pairs/` — CSV + manifest
- `figures/adversarial_resume_pairs/` — coverage / sanity plots

**Principles:** Only edit cues approved in the spec (`city_location`, `age_cue`, `gendered_cue`, `name_optional`). Preserve employers, skills, timelines, and responsibilities. One primary sensitive attribute per pair; metadata documents every change.


## Pair ID convention

- `anchor_row_id`: stable row key in the English test split (`english_test_{ordinal:06d}`).
- `pair_id`: `p_` + first 16 hex chars of SHA256(`anchor_row_id|attribute_edited|transform_type`).
- `variant_id`: same as `pair_id` for single-step transforms (one counterfactual per row in this registry).

Downstream notebooks (**57–58**) should join on `pair_id` / `anchor_row_id`.


In [1]:
"""Load benchmark spec (55) and English test split."""
from __future__ import annotations

import json
import re
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

_CWD = Path.cwd().resolve()
NOTEBOOK_DIR = _CWD if _CWD.name == "notebooks" else (_CWD / "notebooks")
REPO_ROOT = NOTEBOOK_DIR.parent
RESULTS_DIR = REPO_ROOT / "notebooks" / "results" / "adversarial_resume_pairs"
FIG_DIR = REPO_ROOT / "figures" / "adversarial_resume_pairs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SPEC_PATH = REPO_ROOT / "notebooks" / "results" / "adversarial_eval_design" / "adversarial_benchmark_spec.json"
if not SPEC_PATH.is_file():
    raise FileNotFoundError(f"Run 55 first; missing spec: {SPEC_PATH}")

spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
ss = spec["source_split"]
TEST_CSV = REPO_ROOT / ss["csv_path_relative"]
TEXT_COL = ss["text_col"]
LABEL_COL = ss["label_col"]

print("Spec notebook:", spec.get("notebook"))
print("Test CSV:", TEST_CSV)

if not TEST_CSV.is_file():
    raise FileNotFoundError(f"Test split missing: {TEST_CSV}. Align data per 32.")

test_df = pd.read_csv(TEST_CSV)
for col in (TEXT_COL, LABEL_COL):
    if col not in test_df.columns:
        raise KeyError(f"Expected column {col!r}; got {list(test_df.columns)}")

test_df = test_df.reset_index(drop=True)
test_df["anchor_row_id"] = test_df.index.map(lambda i: f"english_test_{i:06d}")
test_df["_resume"] = test_df[TEXT_COL].fillna("").astype(str)
test_df["_label"] = test_df[LABEL_COL].astype(str)

PAIR_COLS = spec["pair_registry_columns"]
APPROVED_ATTRS = set(spec.get("sensitive_attributes_v1", []))
print("Approved sensitive attributes:", sorted(APPROVED_ATTRS))
print("Pair columns:", PAIR_COLS)


Spec notebook: 55_adversarial_eval_design.ipynb
Test CSV: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/data/processed/english_dataset_v1/test.csv
Approved sensitive attributes: ['age_cue', 'city_location', 'gendered_cue', 'name_optional']
Pair columns: ['pair_id', 'anchor_row_id', 'variant_id', 'split', 'supercategory', 'attribute_edited', 'transform_type', 'anchor_resume_text', 'counterfactual_resume_text', 'diff_summary', 'edit_rationale', 'validation_status', 'provenance_note']


In [2]:
"""Conservative transform helpers (aligned with 32 city list where possible)."""

CITY_PATTERN = re.compile(
    r"\b("
    r"new york|san francisco|los angeles|seattle|austin|chicago|boston|atlanta|dallas|houston|"
    r"london|berlin|paris|madrid|toronto|vancouver|sydney|melbourne|singapore|dubai|"
    r"bangalore|bengaluru|mumbai|delhi|hyderabad|pune"
    r")\b",
    flags=re.IGNORECASE,
)
SWAP_CITIES = ["new york", "san francisco", "london", "berlin", "singapore"]

# Age: explicit labeled age or "N years old" (not "N years of experience" — require 'old')
AGE_LABEL_RE = re.compile(r"(?i)\bage\s*:?\s*(\d{1,2})\b")
YEARS_OLD_RE = re.compile(r"(?i)(\d{1,2})\s*years?\s+old\b")

# Name: first few lines, two title-case tokens, denylisted job-title prefixes
_TITLE_DENY = {
    "senior", "junior", "staff", "lead", "principal", "software", "data", "product",
    "engineer", "engineering", "manager", "director", "consultant", "developer",
    "analyst", "scientist", "architect", "designer", "resume", "curriculum", "vitae",
}
NAME_LINE_RE = re.compile(r"^#*\s*([A-Z][a-z]+)\s+([A-Z][a-z]+)\s*$")

# Gendered pronouns (word-boundary; single pass male-leaning -> female, else inverse)
MALE_PRON = re.compile(r"\b(He|he|Him|him|His|his)\b")
def extract_city_mentions(text: str) -> list[str]:
    if not text:
        return []
    return sorted({m.group(0).lower() for m in CITY_PATTERN.finditer(text)})


def swap_cities_unified(text: str, target_city: str) -> str:
    if not text:
        return text

    def repl(match: re.Match) -> str:
        original = match.group(0)
        if original.isupper():
            return target_city.upper()
        if original[:1].isupper():
            return target_city.title()
        return target_city.lower()

    return CITY_PATTERN.sub(repl, text)


def shift_age_tokens(text: str) -> tuple[str, str] | None:
    """Return (new_text, summary) or None if no conservative edit applied."""
    m1 = AGE_LABEL_RE.search(text)
    m2 = YEARS_OLD_RE.search(text)
    if m1:
        n = int(m1.group(1))
        n2 = min(n + 4, 75) if n < 60 else max(n - 4, 22)
        new_t = AGE_LABEL_RE.sub(lambda m: m.group(0).replace(m.group(1), str(n2)), text, count=1)
        return new_t, f"age_label {n}->{n2}"
    if m2:
        n = int(m2.group(1))
        n2 = min(n + 3, 70) if n < 55 else max(n - 3, 21)
        new_t = YEARS_OLD_RE.sub(f"{n2} years old", text, count=1)
        return new_t, f"years_old {n}->{n2}"
    return None


_MALE_TO_FEM = {"He": "She", "he": "she", "Him": "Her", "him": "her", "His": "Her", "his": "her"}


def male_to_female_pronouns(text: str) -> tuple[str, int] | None:
    if not MALE_PRON.search(text):
        return None

    def repl(m: re.Match) -> str:
        return _MALE_TO_FEM[m.group(1)]

    new_t, n = MALE_PRON.subn(repl, text)
    return new_t, n


def try_header_name_swap(text: str) -> tuple[str, str, str] | None:
    """If a plausible personal name line is found early, swap first name only."""
    lines = text.splitlines()
    for line in lines[:12]:
        s = line.strip()
        m = NAME_LINE_RE.match(s)
        if not m:
            continue
        a, b = m.group(1), m.group(2)
        if a.lower() in _TITLE_DENY or b.lower() in _TITLE_DENY:
            continue
        alt_first = "Alex" if a != "Alex" else "Jordan"
        new_line = line.replace(f"{a} {b}", f"{alt_first} {b}", 1)
        new_text = text.replace(line, new_line, 1)
        return new_text, f"{a} {b}", f"{alt_first} {b}"
    return None


def pair_hash(anchor_id: str, attr: str, transform_type: str) -> str:
    h = hashlib.sha256(f"{anchor_id}|{attr}|{transform_type}".encode("utf-8")).hexdigest()[:16]
    return f"p_{h}"


In [3]:
"""Emit pair rows (one transform per pair; multiple pairs per anchor allowed)."""
rows_out: list[dict] = []
provenance = {
    "generator_notebook": "56_adversarial_resume_pair_generation.ipynb",
    "generator_version": "v1_rule_based",
    "spec_path": str(SPEC_PATH.relative_to(REPO_ROOT)),
    "utc_generated": datetime.now(timezone.utc).isoformat(),
    "approved_attributes": sorted(APPROVED_ATTRS),
}

for _, r in test_df.iterrows():
    anchor_id = r["anchor_row_id"]
    anchor_text = r["_resume"]
    label = r["_label"]
    if not anchor_text.strip():
        continue

    # 1) City — one pair: unify all city mentions to first eligible swap city
    if "city_location" in APPROVED_ATTRS:
        cities = extract_city_mentions(anchor_text)
        if cities:
            source_city = cities[0]
            target = next((c for c in SWAP_CITIES if c != source_city), None)
            if target is None:
                target = "toronto" if source_city != "toronto" else "berlin"
            cf = swap_cities_unified(anchor_text, target)
            if cf != anchor_text:
                tt = f"city_unify_to_{target.replace(' ', '_')}"
                pid = pair_hash(anchor_id, "city_location", tt)
                rows_out.append(
                    {
                        "pair_id": pid,
                        "anchor_row_id": anchor_id,
                        "variant_id": pid,
                        "split": ss["split_name"],
                        "supercategory": label,
                        "attribute_edited": "city_location",
                        "transform_type": tt,
                        "anchor_resume_text": anchor_text,
                        "counterfactual_resume_text": cf,
                        "diff_summary": f"Unified city mentions toward {target} (from {source_city}…)",
                        "edit_rationale": "Geographic substitution only; same employers/skills per 55. Pattern mirrors 32 CITY_PATTERN.",
                        "validation_status": "validated",
                        "provenance_note": json.dumps({**provenance, "rule": "city_unify"}),
                    }
                )

    # 2) Age cues
    if "age_cue" in APPROVED_ATTRS:
        age_res = shift_age_tokens(anchor_text)
        if age_res:
            cf, summ = age_res
            tt = "age_numeric_shift_conservative"
            pid = pair_hash(anchor_id, "age_cue", tt)
            rows_out.append(
                {
                    "pair_id": pid,
                    "anchor_row_id": anchor_id,
                    "variant_id": pid,
                    "split": ss["split_name"],
                    "supercategory": label,
                    "attribute_edited": "age_cue",
                    "transform_type": tt,
                    "anchor_resume_text": anchor_text,
                    "counterfactual_resume_text": cf,
                    "diff_summary": summ,
                    "edit_rationale": "Adjusted explicit age wording only; employment dates untouched.",
                    "validation_status": "validated",
                    "provenance_note": json.dumps({**provenance, "rule": "age_cue"}),
                }
            )

    # 3) Gendered cues (conservative: male-coded pronouns -> female-coded only; avoids possessive ambiguity)
    if "gendered_cue" in APPROVED_ATTRS:
        g = male_to_female_pronouns(anchor_text)
        if g is not None:
            cf, nsub = g
            if cf != anchor_text:
                tt = f"pronoun_swap_n{nsub}"
                pid = pair_hash(anchor_id, "gendered_cue", tt)
                rows_out.append(
                    {
                        "pair_id": pid,
                        "anchor_row_id": anchor_id,
                        "variant_id": pid,
                        "split": ss["split_name"],
                        "supercategory": label,
                        "attribute_edited": "gendered_cue",
                        "transform_type": tt,
                        "anchor_resume_text": anchor_text,
                        "counterfactual_resume_text": cf,
                        "diff_summary": f"Replaced {nsub} gendered pronoun token(s); duties unchanged.",
                        "edit_rationale": "Surface pronoun edit only; avoid rewriting job content.",
                        "validation_status": "validated",
                        "provenance_note": json.dumps({**provenance, "rule": "gendered_cue"}),
                    }
                )

    # 4) Optional name (header line)
    if "name_optional" in APPROVED_ATTRS:
        nm = try_header_name_swap(anchor_text)
        if nm:
            cf, before, after = nm
            tt = "header_first_name_substitution_neutral_pool"
            pid = pair_hash(anchor_id, "name_optional", tt)
            rows_out.append(
                {
                    "pair_id": pid,
                    "anchor_row_id": anchor_id,
                    "variant_id": pid,
                    "split": ss["split_name"],
                    "supercategory": label,
                    "attribute_edited": "name_optional",
                    "transform_type": tt,
                    "anchor_resume_text": anchor_text,
                    "counterfactual_resume_text": cf,
                    "diff_summary": f"Header name '{before}' -> '{after}' (first name only).",
                    "edit_rationale": "Name tokens on early line only; reduced stereotype charge via neutral alternates.",
                    "validation_status": "validated",
                    "provenance_note": json.dumps({**provenance, "rule": "name_optional"}),
                }
            )

pairs_df = pd.DataFrame(rows_out)
if len(pairs_df):
    pairs_df = pairs_df[[c for c in PAIR_COLS if c in pairs_df.columns]]

out_csv = RESULTS_DIR / "adversarial_resume_pairs.csv"
pairs_df.to_csv(out_csv, index=False)

summary = {
    "n_test_rows": int(len(test_df)),
    "n_pairs": int(len(pairs_df)),
    "counts_by_attribute": pairs_df.groupby("attribute_edited").size().to_dict() if len(pairs_df) else {},
    "counts_by_class": pairs_df.groupby("supercategory").size().to_dict() if len(pairs_df) else {},
    "output_csv": str(out_csv.relative_to(REPO_ROOT)),
}
(RESULTS_DIR / "pair_generation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(RESULTS_DIR / "pair_generation_manifest.json").write_text(
    json.dumps({**provenance, "summary": summary}, indent=2),
    encoding="utf-8",
)

print("Wrote", out_csv)
print(json.dumps(summary, indent=2))


Wrote /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/results/adversarial_resume_pairs/adversarial_resume_pairs.csv
{
  "n_test_rows": 507,
  "n_pairs": 208,
  "counts_by_attribute": {
    "city_location": 208
  },
  "counts_by_class": {
    "backend_general_dev": 67,
    "generic_it_ops": 9,
    "it_governance_leadership": 1,
    "project_product": 17,
    "sysadmin_devops_network": 25,
    "tech_support_helpdesk": 11,
    "technical_specialized": 46,
    "web_frontend": 32
  },
  "output_csv": "notebooks/results/adversarial_resume_pairs/adversarial_resume_pairs.csv"
}


In [4]:
"""Simple coverage figure (requires matplotlib)."""
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    if len(pairs_df):
        pairs_df["attribute_edited"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4c78a8")
        axes[0].set_title("Pairs by attribute_edited")
        axes[0].tick_params(axis="x", rotation=30)
        pairs_df["supercategory"].value_counts().head(15).plot(kind="barh", ax=axes[1], color="#f58518")
        axes[1].set_title("Top supercategories (pair rows)")
    else:
        axes[0].text(0.5, 0.5, "No pairs generated", ha="center")
    fig.suptitle("56 — Adversarial pair coverage")
    fig.tight_layout()
    fp = FIG_DIR / "pair_counts_overview.png"
    fig.savefig(fp, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("Figure:", fp)
except Exception as e:
    print("Skipped figure:", e)


Figure: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/adversarial_resume_pairs/pair_counts_overview.png


## Key takeaways

1. **What was generated:** Rule-based **counterfactual resume texts** on the **English test split** only, with separate rows for each approved attribute family (`city_location`, `age_cue`, `gendered_cue`, `name_optional`) when the conservative detectors fired.
2. **Constraints:** No invented employers or skills; **supercategory is unchanged**; one primary attribute edit per pair; **stable IDs** tie each counterfactual to an `anchor_row_id`.
3. **For downstream:** Notebook **57** should load `adversarial_resume_pairs.csv` for **QA / audit**. Notebook **58** should run **existing checkpoints** on anchor vs counterfactual columns and report **flip rate** and **score deltas** — **no retraining** and **no regeneration** of pairs in 58.
